# OpenCV Crash Course: 2026 Edition
## Module 6: Tracking & Counting
**Estimated time: 15 minutes**

- Give every object an ID with YOLO + ByteTrack
- Count objects crossing a line (vehicles, people, products on a conveyor)

> Detection says "5 cars". Tracking says "car #12 crossed the line".

In [ ]:
# !pip install ultralytics "supervision[assets]"

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
import supervision as sv
from supervision.assets import download_assets, VideoAssets

def show(img, title="", figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.show()

os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)
VIDEO = 'input/vehicles.mp4'
if not os.path.exists(VIDEO):
    os.replace(download_assets(VideoAssets.VEHICLES), VIDEO)   # free sample traffic video
info  = sv.VideoInfo.from_video_path(VIDEO)
print(info)

VEHICLES = [2, 3, 5, 7]   # COCO ids: car, motorcycle, bus, truck

## 1. Tracking in One Line

`model.track(frame, persist=True)` = detection + tracker. Each object keeps the same **ID** across frames.

In [ ]:
model = YOLO('models/yolo26n.pt')
cap = cv2.VideoCapture(VIDEO)
for i in range(30):
    ok, frame = cap.read()
    r = model.track(frame, persist=True, tracker='bytetrack.yaml',
                    classes=VEHICLES, conf=0.3, verbose=False)[0]
cap.release()

print("Track IDs:", r.boxes.id.int().tolist())
show(r.plot(), 'Each vehicle has an ID')

## 2. Line-Crossing Counter

`supervision` (by Roboflow) gives a ready line counter and annotators.

In [ ]:
model = YOLO('models/yolo26n.pt')                       # fresh tracker
W, H = info.resolution_wh
line     = sv.LineZone(start=sv.Point(0, int(H * 0.55)), end=sv.Point(W, int(H * 0.55)))
line_ann = sv.LineZoneAnnotator(thickness=3, text_scale=1.2)
box_ann  = sv.BoxAnnotator(thickness=2)
lbl_ann  = sv.LabelAnnotator(text_scale=0.6)

with sv.VideoSink('output/vehicles_counted.mp4', info) as sink:
    for frame in sv.get_video_frames_generator(VIDEO, end=150):
        r = model.track(frame, persist=True, tracker='bytetrack.yaml',
                        classes=VEHICLES, conf=0.3, verbose=False)[0]
        det = sv.Detections.from_ultralytics(r)
        out = frame.copy()
        if det.tracker_id is not None:
            line.trigger(det)
            out = box_ann.annotate(out, det)
            out = lbl_ann.annotate(out, det, [f"#{t}" for t in det.tracker_id])
        out = line_ann.annotate(out, line_counter=line)
        sink.write_frame(out)

print(f"IN: {line.in_count}  OUT: {line.out_count}   saved: output/vehicles_counted.mp4")
show(out, f'IN={line.in_count}  OUT={line.out_count}')

---
## Summary

| Task | Code |
|------|------|
| Track | `model.track(frame, persist=True, tracker='bytetrack.yaml')` |
| IDs | `r.boxes.id` |
| To supervision | `sv.Detections.from_ultralytics(r)` |
| Line counter | `sv.LineZone(start, end)`, `.trigger(det)`, `.in_count` |
| Zone counter | `sv.PolygonZone(polygon).trigger(det)` |

**Next:** Module 7, Train YOLO on your own objects